# Unlearning annotation pipeline

Code sample for the RA application. This notebook downloads two PDFs from the document repository, extracts paragraph-level units with PyMuPDF, builds prompts from the unlearning codebook, calls the OpenAI API, parses structured JSON outputs, and writes results to CSV. It also includes a small benchmark against available coded examples.


In [1]:
# Run once if needed
# %pip install pymupdf pandas openpyxl requests tqdm openai scikit-learn

In [2]:
from pathlib import Path
import os, re, json, time
from datetime import datetime
from openai import OpenAI


import pandas as pd
import requests
import pymupdf  # PyMuPDF
from tqdm import tqdm
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

BASE_DIR = Path('.')
DATA_DIR = BASE_DIR / 'data'
PDF_DIR = BASE_DIR / 'pdfs'
OUT_DIR = BASE_DIR / 'outputs'
for d in [DATA_DIR, PDF_DIR, OUT_DIR]:
    d.mkdir(exist_ok=True)

REPO_PATH = DATA_DIR / 'Dataset_17Nov_2025_v1.xlsx'
CODEBOOK_PATH = DATA_DIR / 'Unlearning Codebook_6April2026_v1.xlsx'

# If you run this notebook outside the zip folder, update these two paths.
assert REPO_PATH.exists(), f'Missing {REPO_PATH}'
assert CODEBOOK_PATH.exists(), f'Missing {CODEBOOK_PATH}'


**API key note:** I executed the notebook using my own OpenAI API key. To reproduce the LLM calls, please use your own `OPENAI_API_KEY` before running the notebook.

In [3]:
# Configuration
N_DOCS = 2
MAX_PARAGRAPHS_FOR_LLM = 10000  # run all extracted paragraphs from the selected PDFs
MAX_GOLD_ROWS_FOR_LLM = 10000   # run all available gold examples
MIN_TOKENS = 25
MAX_TOKENS = 350

OPENAI_MODEL = 'gpt-4o-mini'    # change if needed
TEMPERATURE = 0
MAX_OUTPUT_TOKENS = 350
PROMPT_VERSION = 'v0_quick_demo'

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', '') #Add API Key Here 

SEED = 42

RUN_LLM = bool(OPENAI_API_KEY)
print('RUN_LLM =', RUN_LLM)


RUN_LLM = True


## 1. Load repository and codebook

In [4]:
repo = pd.read_excel(REPO_PATH)
repo = repo.rename(columns=lambda c: str(c).strip())
repo['doc_id'] = [f'DOC_{i+1:03d}' for i in range(len(repo))]
repo['Title'] = repo['Title'].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
repo['Agency'] = repo['Agency'].fillna('Unknown').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
repo['Year'] = pd.to_numeric(repo['Year'], errors='coerce').astype('Int64')
repo['Link'] = repo['Link'].astype(str).str.strip()
repo['is_direct_pdf'] = repo['Link'].str.lower().str.endswith('.pdf')

sample_docs = (
    repo[repo['is_direct_pdf'] & repo['Page length'].between(2, 50, inclusive='both')]
    .sort_values('Page length')
    .head(N_DOCS)
    .reset_index(drop=True)
)

sample_docs[['doc_id', 'Title', 'Agency', 'Year', 'Page length', 'Link']]


,doc_id,Title,Agency,Year,Page length,Link
0,DOC_021,U.S. Army Corps of Engineers releases its “12 ...,USACE,2006,2.0,https://corpslakes.erdc.dren.mil/employees/cus...
1,DOC_031,Statement by Comptroller General David M. Walk...,GAO,2006,9.0,https://www.gao.gov/assets/gao-06-365r.pdf


In [5]:
raw_cb = pd.read_excel(CODEBOOK_PATH, sheet_name='Codebook (revised)', header=None)
headers = raw_cb.iloc[1].tolist()
codebook = raw_cb.iloc[2:].copy()
codebook.columns = headers
codebook = codebook.dropna(how='all')
codebook = codebook[codebook['Code'].notna()].copy()
codebook['Code'] = codebook['Code'].astype(str).str.strip()

# keep the useful rows for the prompt
prompt_rows = codebook[~codebook['Code'].str.upper().str.startswith(('STEP', 'CODING', 'REVISED'))].copy()
prompt_rows[['Code', 'Definition', 'Detection Logic']].head(10)


,Code,Definition,Detection Logic
3,Unlearning (Yes/No),Unlearning is the deliberate process by which ...,Does the text explicitly identify a prior poli...
5,Target (Leadership),"The text questions or redefines the role, auth...",Does the text call for redefining or redistrib...
6,"Target (laws, plans and policies)","The text calls for terminating, replacing, or ...","Does the text identify a specific law, plan, o..."
7,Target (capabilities),The text identifies a technical system — model...,Does the text identify a specific technical sy...
8,Target (funds and resource allocation),The text calls for changing how disaster funds...,Does the text identify the existing funds mana...
9,"Target (misclleanous) organizational culture, ...",The text calls for unlearning in a domain not ...,Does the text call for discarding a prior assu...
11,Government Agency,"The specific federal agency, sub-unit, or inst...","Does the text name a specific agency, office, ..."
13,Unit of Analysis,Code at the paragraph level. Do not split a pa...,Does the paragraph as a whole constitute an in...
14,ICA Protocol,"Each coder assigns: (1) Unlearning Yes/No, (2)...",—


## 2. Download two PDFs and extract paragraph-level text

In [6]:
def safe_filename(doc_id, title):
    title = re.sub(r'[^A-Za-z0-9]+', '_', str(title))[:55].strip('_')
    return f'{doc_id}_{title}.pdf'


def download_pdf(url, out_path, timeout=60):
    if out_path.exists() and out_path.stat().st_size > 1000:
        return out_path
    r = requests.get(url, timeout=timeout, headers={'User-Agent': 'Mozilla/5.0'})
    r.raise_for_status()
    out_path.write_bytes(r.content)
    return out_path

pdf_records = []
for _, row in sample_docs.iterrows():
    pdf_path = PDF_DIR / safe_filename(row['doc_id'], row['Title'])
    try:
        download_pdf(row['Link'], pdf_path)
        pdf_records.append({**row.to_dict(), 'pdf_path': str(pdf_path), 'download_status': 'ok'})
    except Exception as e:
        pdf_records.append({**row.to_dict(), 'pdf_path': str(pdf_path), 'download_status': f'failed: {e}'})

pdf_df = pd.DataFrame(pdf_records)
pdf_df[['doc_id', 'Title', 'download_status', 'pdf_path']]


,doc_id,Title,download_status,pdf_path
0,DOC_021,U.S. Army Corps of Engineers releases its “12 ...,ok,pdfs\DOC_021_U_S_Army_Corps_of_Engineers_relea...
1,DOC_031,Statement by Comptroller General David M. Walk...,ok,pdfs\DOC_031_Statement_by_Comptroller_General_...


In [7]:
def clean_paragraph_text(text):
    text = str(text).replace("\u00ad", "")
    text = text.replace("\r", "\n")
    text = re.sub(r"(?<=\w)-\n(?=\w)", "", text)   # de-hyphenate line breaks
    text = re.sub(r"\n+", " ", text)               # join wrapped lines within a paragraph
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_block_into_paragraphs(block_text):
    """
    PyMuPDF blocks are layout blocks, not guaranteed paragraphs.
    This splits a block into paragraph-like units before cleaning.
    """
    block_text = str(block_text).replace("\r", "\n").strip()

    # Split on blank-line boundaries before collapsing whitespace.
    parts = re.split(r"\n\s*\n+", block_text)

    cleaned = []
    for part in parts:
        para = clean_paragraph_text(part)
        if para:
            cleaned.append(para)

    return cleaned


def simple_tokens(text):
    return re.findall(r"\w+|[^\w\s]", str(text), flags=re.UNICODE)


def extract_paragraphs_with_pymupdf(pdf_path, doc_meta):
    rows = []
    pdf_path = Path(pdf_path)

    with pymupdf.open(pdf_path) as doc:
        for page_idx, page in enumerate(doc, start=1):
            blocks = page.get_text("blocks", sort=True)
            page_text_chars = 0
            page_para_number = 1

            for block_idx, block in enumerate(blocks, start=1):
                # PyMuPDF block: x0, y0, x1, y1, text, block_no, block_type
                raw_text = block[4]
                page_text_chars += len(str(raw_text))

                for para in split_block_into_paragraphs(raw_text):
                    tok_count = len(simple_tokens(para))

                    if tok_count < MIN_TOKENS or tok_count > MAX_TOKENS:
                        continue

                    rows.append({
                        "doc_id": doc_meta["doc_id"],
                        "title": doc_meta["Title"],
                        "agency_doc": doc_meta["Agency"],
                        "year": doc_meta["Year"],
                        "source_url": doc_meta["Link"],
                        "page_number": page_idx,
                        "block_number": block_idx,
                        "paragraph_number": page_para_number,
                        "paragraph_id": f"{doc_meta['doc_id']}_p{page_idx:03d}_para{page_para_number:03d}",
                        "text": para,
                        "token_count": tok_count,
                        "needs_ocr": page_text_chars < 50,
                        "pdf_path": str(pdf_path)
                    })

                    page_para_number += 1

    return rows


all_rows = []
for _, row in pdf_df[pdf_df["download_status"].eq("ok")].iterrows():
    all_rows.extend(extract_paragraphs_with_pymupdf(row["pdf_path"], row))

paragraph_columns = [
    "doc_id", "title", "agency_doc", "year", "source_url",
    "page_number", "block_number", "paragraph_number",
    "paragraph_id", "text", "token_count", "needs_ocr", "pdf_path"
]

paragraphs = pd.DataFrame(all_rows, columns=paragraph_columns)
paragraphs.to_csv(OUT_DIR / "extracted_paragraphs.csv", index=False)
paragraphs.head()

,doc_id,title,agency_doc,year,source_url,page_number,block_number,paragraph_number,paragraph_id,text,token_count,needs_ocr,pdf_path
0,DOC_021,U.S. Army Corps of Engineers releases its “12 ...,USACE,2006,https://corpslakes.erdc.dren.mil/employees/cus...,1,7,1,DOC_021_p001_para001,"Washington (August 24, 2006) – The commander o...",49,False,pdfs\DOC_021_U_S_Army_Corps_of_Engineers_relea...
1,DOC_021,U.S. Army Corps of Engineers releases its “12 ...,USACE,2006,https://corpslakes.erdc.dren.mil/employees/cus...,1,7,2,DOC_021_p001_para002,“Hurricane Katrina’s disastrous impact upon th...,67,False,pdfs\DOC_021_U_S_Army_Corps_of_Engineers_relea...
2,DOC_021,U.S. Army Corps of Engineers releases its “12 ...,USACE,2006,https://corpslakes.erdc.dren.mil/employees/cus...,1,7,3,DOC_021_p001_para003,“Exhaustive analysis by the Corps and other in...,49,False,pdfs\DOC_021_U_S_Army_Corps_of_Engineers_relea...
3,DOC_021,U.S. Army Corps of Engineers releases its “12 ...,USACE,2006,https://corpslakes.erdc.dren.mil/employees/cus...,1,7,4,DOC_021_p001_para004,“These 12 actions were developed from that ana...,63,False,pdfs\DOC_021_U_S_Army_Corps_of_Engineers_relea...
4,DOC_021,U.S. Army Corps of Engineers releases its “12 ...,USACE,2006,https://corpslakes.erdc.dren.mil/employees/cus...,1,7,5,DOC_021_p001_para005,The “12 Actions for Change” fall within three ...,35,False,pdfs\DOC_021_U_S_Army_Corps_of_Engineers_relea...


In [8]:
print("PDFs processed:", paragraphs["doc_id"].nunique() if len(paragraphs) else 0)
print("Paragraph-like units extracted:", len(paragraphs))
if len(paragraphs):
    display(paragraphs[["paragraph_id", "title", "page_number", "paragraph_number", "token_count", "text"]].head(5))

PDFs processed: 2
Paragraph-like units extracted: 50


,paragraph_id,title,page_number,paragraph_number,token_count,text
0,DOC_021_p001_para001,U.S. Army Corps of Engineers releases its “12 ...,1,1,49,"Washington (August 24, 2006) – The commander o..."
1,DOC_021_p001_para002,U.S. Army Corps of Engineers releases its “12 ...,1,2,67,“Hurricane Katrina’s disastrous impact upon th...
2,DOC_021_p001_para003,U.S. Army Corps of Engineers releases its “12 ...,1,3,49,“Exhaustive analysis by the Corps and other in...
3,DOC_021_p001_para004,U.S. Army Corps of Engineers releases its “12 ...,1,4,63,“These 12 actions were developed from that ana...
4,DOC_021_p001_para005,U.S. Army Corps of Engineers releases its “12 ...,1,5,35,The “12 Actions for Change” fall within three ...


## 3. Prompt template + JSON schema

In [9]:
def codebook_for_prompt(df):
    parts = []
    for _, r in df.iterrows():
        parts.append(
            f"Code: {r.get('Code', '')}\n"
            f"Definition: {r.get('Definition', '')}\n"
            f"Detection logic: {r.get('Detection Logic', '')}\n"
            f"Positive: {r.get('Positive Clarification', '')}\n"
            f"Negative: {r.get('Negative Clarification', '')}"
        )
    return "\n\n".join(parts)

CODEBOOK_TEXT = codebook_for_prompt(prompt_rows)

JSON_SCHEMA_NOTE = """
Return valid JSON only with this structure:
{
  "unlearning_present": true/false,
  "target_type": "Leadership | laws_plans_policies | capabilities | funds_resources | misc_organizational | none",
  "agency": "agency or sub-unit name if stated, otherwise null",
  "rationale": "one short sentence"
}
""".strip()


def build_prompt(text, paragraph_id=None):
    return f"""
You are coding federal disaster policy text using the unlearning codebook.

Task:
1. Decide whether the paragraph contains unlearning.
2. If yes, assign one target type.
3. Identify the agency or sub-unit if the paragraph states one.
4. Keep the rationale short.

Codebook:
{CODEBOOK_TEXT}

{JSON_SCHEMA_NOTE}

Paragraph ID: {paragraph_id}
Paragraph:
{text}
""".strip()



## 4. LLM API call and JSON parsing

In [10]:
def parse_json_response(text):
    text = str(text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', text, flags=re.S)
        if match:
            return json.loads(match.group(0))
        raise


def call_openai(prompt):
    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.chat.completions.create(
    model=OPENAI_MODEL,
    temperature=TEMPERATURE,
    seed=SEED,
    max_tokens=MAX_OUTPUT_TOKENS,
    response_format={'type': 'json_object'},
    messages=[
        {'role': 'system', 'content': 'Return valid JSON only. Do not include markdown.'},
        {'role': 'user', 'content': prompt}
    ]
    )
    
    return {
    "content": resp.choices[0].message.content,
    "system_fingerprint": getattr(resp, "system_fingerprint", None)
    }


def annotate_one(text, paragraph_id=None):
    prompt = build_prompt(text, paragraph_id)
    started = time.time()
    api_result = call_openai(prompt)
    raw = api_result["content"]
    parsed = parse_json_response(raw)
    
    parsed.update({
        'paragraph_id': paragraph_id,
        'provider': 'openai',
        'model': OPENAI_MODEL,
        'temperature': TEMPERATURE,
        'seed': SEED,
        'system_fingerprint': api_result["system_fingerprint"],
        'prompt_version': PROMPT_VERSION,
        'runtime_seconds': round(time.time() - started, 3),
        'run_timestamp': datetime.now().isoformat(timespec='seconds'),
        'raw_response': raw
    })
    return parsed


## 5. Run on extracted paragraphs

In [11]:
demo_rows = paragraphs.head(MAX_PARAGRAPHS_FOR_LLM).copy()
results = []

if RUN_LLM and len(demo_rows):
    for _, row in tqdm(demo_rows.iterrows(), total=len(demo_rows)):
        try:
            results.append(annotate_one(row['text'], row['paragraph_id']))
        except Exception as e:
            results.append({
                'paragraph_id': row['paragraph_id'],
                'error': str(e),
                'provider': 'openai',
                'model': OPENAI_MODEL,
                'prompt_version': PROMPT_VERSION,
                'run_timestamp': datetime.now().isoformat(timespec='seconds')
            })
else:
    print('LLM calls skipped. Add OPENAI_API_KEY / set env var to run this cell against the PDFs.')

annotations = pd.DataFrame(results)
annotations.to_csv(OUT_DIR / 'llm_annotations_pdf_sample.csv', index=False)
annotations.head()


100%|██████████| 50/50 [01:36<00:00,  1.94s/it]


,unlearning_present,target_type,agency,rationale,paragraph_id,provider,model,temperature,seed,system_fingerprint,prompt_version,runtime_seconds,run_timestamp,raw_response
0,False,none,United States Army Corps of Engineers,The paragraph describes a set of actions for c...,DOC_021_p001_para001,openai,gpt-4o-mini,0,42,fp_21d4bc30a8,v0_quick_demo,3.312,2026-05-15T13:09:32,"{\n ""unlearning_present"": false,\n ""target_t..."
1,False,none,U.S. Army Corps of Engineers,The paragraph reflects on the impact of Hurric...,DOC_021_p001_para002,openai,gpt-4o-mini,0,42,fp_21d4bc30a8,v0_quick_demo,1.869,2026-05-15T13:09:34,"{\n ""unlearning_present"": false,\n ""target_t..."
2,True,Leadership,U.S. Army Corps of Engineers,The text calls for transforming the Corps' app...,DOC_021_p001_para003,openai,gpt-4o-mini,0,42,fp_21d4bc30a8,v0_quick_demo,2.085,2026-05-15T13:09:36,"{\n ""unlearning_present"": true,\n ""target_ty..."
3,False,none,None,The paragraph does not identify any prior poli...,DOC_021_p001_para004,openai,gpt-4o-mini,0,42,fp_21d4bc30a8,v0_quick_demo,1.684,2026-05-15T13:09:38,"{\n ""unlearning_present"": false,\n ""target_t..."
4,False,none,None,The paragraph does not identify any prior poli...,DOC_021_p001_para005,openai,gpt-4o-mini,0,42,fp_21d4bc30a8,v0_quick_demo,1.750,2026-05-15T13:09:39,"{\n ""unlearning_present"": false,\n ""target_t..."


## 6. Mini benchmark against coded examples

In [12]:
gold = pd.read_excel(CODEBOOK_PATH, sheet_name='GPT Test')
gold = gold.rename(columns=lambda c: str(c).strip())
gold = gold[
    gold["Number"].astype(str).str.match(r"^\d+:\d+$", na=False)
].copy()
gold = gold[gold["Anmol_Binary"].isin(["Yes", "No"])].copy()
gold['gold_binary'] = gold['Anmol_Binary'].astype(str).str.lower().map({'yes': True, 'no': False})
gold_sample = gold.head(MAX_GOLD_ROWS_FOR_LLM).copy()

gold_results = []
if RUN_LLM and len(gold_sample):
    for _, row in tqdm(gold_sample.iterrows(), total=len(gold_sample)):
        pred = annotate_one(row['Text Content'], row['Number'])
        pred['gold_binary'] = bool(row['gold_binary']) if pd.notna(row['gold_binary']) else None
        pred['source_document'] = row.get('Document')
        gold_results.append(pred)
else:
    print('Gold benchmark LLM calls skipped. Add API key / set env var to run it.')

gold_pred = pd.DataFrame(gold_results)
gold_pred.to_csv(OUT_DIR / 'gold_sample_predictions.csv', index=False)
gold_pred.head()


100%|██████████| 11/11 [00:19<00:00,  1.80s/it]


,unlearning_present,target_type,agency,rationale,paragraph_id,provider,model,temperature,seed,system_fingerprint,prompt_version,runtime_seconds,run_timestamp,raw_response,gold_binary,source_document
0,True,Leadership,FEMA,The text questions FEMA's organizational place...,14:2,openai,gpt-4o-mini,0,42,fp_21d4bc30a8,v0_quick_demo,1.692,2026-05-15T13:11:08,"{\n ""unlearning_present"": true,\n ""target_ty...",True,gao-06-442t.pdf
1,True,Leadership,FEMA,The text calls for designating a single indivi...,14:3,openai,gpt-4o-mini,0,42,fp_21d4bc30a8,v0_quick_demo,1.880,2026-05-15T13:11:10,"{\n ""unlearning_present"": true,\n ""target_ty...",True,gao-06-442t.pdf
2,True,Leadership,FEMA,The text questions FEMA's organizational place...,14:4,openai,gpt-4o-mini,0,42,fp_21d4bc30a8,v0_quick_demo,1.568,2026-05-15T13:11:11,"{\n ""unlearning_present"": true,\n ""target_ty...",True,gao-06-442t.pdf
3,True,capabilities,Federal government,The text calls for a departure from the tradit...,14:5,openai,gpt-4o-mini,0,42,fp_21d4bc30a8,v0_quick_demo,2.109,2026-05-15T13:11:13,"{\n ""unlearning_present"": true,\n ""target_ty...",True,gao-06-442t.pdf
4,False,none,FEMA,The paragraph discusses past recommendations w...,14:6,openai,gpt-4o-mini,0,42,fp_21d4bc30a8,v0_quick_demo,1.645,2026-05-15T13:11:15,"{\n ""unlearning_present"": false,\n ""target_t...",True,gao-06-442t.pdf


In [13]:
def to_bool(x):
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return None
    return str(x).strip().lower() in ['true', 'yes', '1', 'y']

metrics_rows = []
if len(gold_pred) and {'gold_binary', 'unlearning_present'}.issubset(gold_pred.columns):
    eval_df = gold_pred.dropna(subset=['gold_binary', 'unlearning_present']).copy()
    eval_df['pred_binary'] = eval_df['unlearning_present'].map(to_bool)
    y_true = eval_df['gold_binary'].astype(bool)
    y_pred = eval_df['pred_binary'].astype(bool)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    metrics_rows.append({
        'provider': 'openai',
        'model': OPENAI_MODEL,
        'n': len(eval_df),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'prompt_version': PROMPT_VERSION
    })

metrics = pd.DataFrame(metrics_rows)
metrics.to_csv(OUT_DIR / 'eval_metrics.csv', index=False)
metrics


,provider,model,n,accuracy,precision,recall,f1,prompt_version
0,openai,gpt-4o-mini,11,0.727273,1.0,0.727273,0.842105,v0_quick_demo


## 7. Files produced

In [14]:
for p in sorted(OUT_DIR.glob('*.csv')):
    print(p, p.stat().st_size, 'bytes')


outputs\eval_metrics.csv 160 bytes
outputs\extracted_paragraphs.csv 42897 bytes
outputs\gold_sample_predictions.csv 6163 bytes
outputs\llm_annotations_pdf_sample.csv 22961 bytes
